# 代码生成器

需求：使用 Frontier 模型，从 Python 代码生成高性能的 C++ 代码



<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">提醒：拉取最新代码</h2>
            <span style="color:#f71;">我在持续改进这些实验，添加更多示例和练习。
            每周开始时，值得检查一下你是否拥有最新代码。<br/>
            首先进行 <a href="https://chatgpt.com/share/6734e705-3270-8012-a074-421661af6ba9">git pull 并按需合并你的更改</a>。有问题？可以请 ChatGPT 说明如何合并——或者联系我！<br/><br/>
            拉取代码后，在 llm_engineering 目录下，于 Cursor Terminal 中运行：<br/>
            <code>uv sync</code><br/>
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">重要提示</h1>
            <span style="color:#900;">
            在本实验中，我使用高端模型 GPT 5、Claude 4.5 Sonnet、Gemini 2.5 Pro、Grok 4，这些是价格稍高的模型。费用仍然很低，但如果你更希望把成本压到极低，请选择像 gpt-5-nano 这样的低成本模型。
            </span>
        </td>
    </tr>
</table>

In [ ]:
# 导入

# 导入标准库 os（操作系统相关，用来读环境变量 Environment Variables）
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：用它调用 Chat Completions 等 API（Application Programming Interface）
from openai import OpenAI
# 导入 subprocess：在 Python 里启动外部命令（如编译器）
import subprocess
# 从 IPython.display 导入展示工具：在 Jupyter 笔记本里漂亮地显示 Markdown/图片等
from IPython.display import Markdown, display

In [ ]:
# 加载 .env 文件：把 API Key 等密钥读入进程环境（override=True 表示覆盖已有同名变量）
load_dotenv(override=True)
# 用 os.getenv 读取环境变量里的密钥；找不到时返回 None
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

In [ ]:
# 连接到客户端库

# 创建 OpenAI 客户端；不传参时默认读环境变量里的 API Key
openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"

# 用 OpenAI 兼容接口连接其它厂商：关键指定 base_url（服务地址）和 api_key
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)

In [ ]:
OPENAI_MODEL = "gpt-5"
CLAUDE_MODEL = "claude-sonnet-4-5-20250929"
GROK_MODEL = "grok-4"
GEMINI_MODEL = "gemini-2.5-pro"

# 想把成本压到极低？取消下面几行的注释：

# OPENAI_MODEL = "gpt-5-nano"
# CLAUDE_MODEL = "claude-haiku-4-5"
# GROK_MODEL = "grok-4-fast-non-reasoning"
# GEMINI_MODEL = "gemini-2.5-flash-lite"

## 请注意：

我们将编写一个解决方案，把 Python 转换成高效、优化的 C++ 代码，以便在你的机器上编译成原生机器码并执行。

你并不需要自己去执行这些代码——那不是本练习的重点！

但如果你想做（因为很有成就感！），我把步骤写在这里。完全可选！

作为替代，我还会给你看一个可以运行 C++ 代码的网站。

In [ ]:
# 导入 system_info：读取本机 CPU/编译器等信息，方便后面生成优化代码
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

In [ ]:
# 准备发给模型的消息文本（message）
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

# 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
response = openai.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": message}])
# 用 Markdown 在笔记本中渲染格式化文本（display_id 方便后续原地刷新）
display(Markdown(response.choices[0].message.content))
    

## 如果你需要安装某些东西

如果你想安装，请按照 GPT 的说明操作！然后重新运行分析（你可能需要 Restart notebook）以确认环境已就绪。

现在你应该已经掌握了编译代码的命令，以及运行它的命令！

请在下方单元格中输入：

In [ ]:
# 准备编译命令参数列表（稍后交给 subprocess 执行）
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

## 接下来，进入主要任务

In [ ]:
# 系统提示词（system prompt）：给模型设定角色与规则，通常用户看不到
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

# 根据 Website 对象生成「请摘要此网页」的用户提示词
def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [ ]:
# 把系统提示词 + 用户内容打包成 API 需要的 messages 结构
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [ ]:
# 把模型生成的代码写入本地源文件，供后续编译运行
def write_output(cpp):
    # 打开文件写入；with 结束时自动关闭文件句柄
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [ ]:
# 把 Python 代码「移植」成高性能目标语言：调模型生成代码并写入文件
def port(client, model, python):
    # reasoning_effort：部分推理模型可调的「思考强度」（如 high），影响耗时与质量
    reasoning_effort = "high" if 'gpt' in model else None
    # 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    # 从响应里取出第一条候选的 message.content（模型生成的文本）
    reply = response.choices[0].message.content
    # 去掉模型回复里的 Markdown 代码围栏（```），只保留纯代码
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [ ]:
# 待移植的 Python 示例代码（字符串形式保存，稍后发给模型）
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [ ]:
# 在受控环境里执行 Python 片段，方便和编译后的结果对比耗时
def run_python(code):
    # 准备 exec 的全局命名空间；限制内建可减少意外副作用（演示用）
    globals = {"__builtins__": __builtins__}
    # exec：把字符串当 Python 代码执行（仅用于可信的课程示例）
    exec(code, globals)

In [ ]:
run_python(pi)

In [ ]:
port(openai, OPENAI_MODEL, pi)

# 编译 C++ 并执行

下一个单元格包含根据 GPT 说明编译 C++ 文件的命令。

同样，如果你不想做这一步，也完全没关系！

或者作为替代：同学 Sandeep K.G. 指出，你可以在线运行 Python 和 C++ 代码来测试。谢谢 Sandeep！  
> 这不是精确对比，但你仍然能感受到性能差异。  
> 例如这里：https://www.programiz.com/cpp-programming/online-compiler/

In [ ]:
# 使用来自 GPT 5 的命令

def compile_and_run():
    # subprocess.run：执行外部命令；check=True 表示失败就抛异常
    subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)

In [ ]:
compile_and_run()

In [ ]:
19.178207/0.082168

## 好，让我们试试其他竞争者！

In [ ]:
port(anthropic, CLAUDE_MODEL, pi)
compile_and_run()

In [ ]:
port(grok, GROK_MODEL, pi)
compile_and_run()

In [ ]:
port(gemini, GEMINI_MODEL, pi)
compile_and_run()


In [ ]:
print(f"""
In Ed's experiments, the performance speedups were:

4th place: Claude Sonnet 4.5: {19.178207/0.104241:.0f}X speedup
3rd place: GPT-5: {19.178207/0.082168:.0f}X speedup
2nd place: Grok 4: {19.178207/0.018092:.0f}X speedup
1st place: Gemini 2.5 Pro: {19.178207/0.013314:.0f}X speedup
""")